In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd drive/MyDrive/jax-f16-gcas-validation

/content/drive/MyDrive/jax-f16-gcas-validation


In [5]:
!pip install jaxlib
!pip install jax_f16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.3 MB/s eta 0:00:00


In [6]:
!pip install --upgrade jax[cuda]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 23.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82


In [7]:
import scipy as sp
from jax.scipy.stats.multivariate_normal import logpdf as multivar_gauss_logpdf


In [8]:
from simulate import *
from f16system import *
import matplotlib.pyplot as plt
import numpy as np

In [9]:
def generate_proposal_distributions(p_mean, p_std, num_proposals=1, scale=1):
    """
    generate proposal distributions by taking the nominal distribution
    parameters and perturbing them by a random value
    """
    qs = [] # proposal distributions
    mask = np.zeros(16) # which components we are going to change for proposal
    mask[3] = 1
    mask[6] = 1
    # create proposal distributions by slightly perturbing the parameters of nominal dist
    for i in range(num_proposals):
        # use mask to only change the noisy features
        q_mean = p_mean + (scale * np.random.randn(16) * mask)
        q_std = p_std # only change the mean for now
        # qs.append(jsp.stats.multivariate_normal(q_mean, system.noise_std))
        qs.append((q_mean, q_std))
    return qs

In [10]:
system = F16System(T=300)
nominal_mean = system.noise_mean
nominal_std = system.noise_std
# p = (nominal_mean, nominal_cov)

In [11]:
print(system.noise_std)

[1.e-10 1.e-10 1.e-10 1.e-02 1.e-10 1.e-10 3.e-02 1.e-10 1.e-10 1.e-10
 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10]


In [12]:
print(jnp.square(system.noise_std))

[1.00000005e-20 1.00000005e-20 1.00000005e-20 9.99999975e-05
 1.00000005e-20 1.00000005e-20 8.99999985e-04 1.00000005e-20
 1.00000005e-20 1.00000005e-20 1.00000005e-20 1.00000005e-20
 1.00000005e-20 1.00000005e-20 1.00000005e-20 1.00000005e-20]


In [13]:
qs = generate_proposal_distributions(nominal_mean, nominal_std, num_proposals=1, scale=1e-3)
# qs = generate_proposal_distributions(nominal_mean, nominal_std, num_proposals=2, scale=3e-1)

In [14]:
def imp_sampl_fail_est(p, qs, num_rollouts, DMMIS=False):
    q_systems = []
    for q in qs:
        q_system = F16System(T=300, prop_noise_mean=q[0], prop_noise_std=q[1])
        q_systems.append(q_system)
    rollouts = []
    ws = []

    if DMMIS:
        print("Not implemented")
        return 0.0
        # for q in tqdm(q_systems, desc="Running Rollouts"):
        #     for j in range(num_rollouts):
        #         rollouts.append(q.rollout_min_altitude_and_loglik())
        # for rollout in tqdm(rollouts, desc="Calculating weights"):
        #     traj = rollout[2][2].reshape(-1, 16)
        #     p_i = gaussian_log_pdf_traj(traj, p.noise_mean, p.noise_cov)
        #     print("\np_i:", p_i)
        #     denom = 0
        #     for q in qs:
        #         denom += gaussian_log_pdf_traj(traj, q[0], np.diag(np.square(q[1])))
        #     # print(denom)
        #     denom /= len(qs)
        #     # else:
        #     #     denom = rollout[1]
        #     print("\nDenom:", denom)
        #     ws.append(p_i / denom)

    else:
        for q in tqdm(q_systems, desc="Running Rollouts"):
            for j in range(num_rollouts):
                rollout = q.rollout_min_altitude_and_loglik()
                rollouts.append(rollout)
                traj = rollout[2][2].reshape(-1, 16)
                p_i = gaussian_log_pdf_traj(traj, p.noise_mean, p.noise_cov)
                q_i = gaussian_log_pdf_traj(traj, q.sensor.mean, q.sensor.cov)
                ws.append(np.exp(p_i - q_i))
    print("Finished proposal distribution likelihood")
    print(ws)
    weighted_sum = 0
    for i in range(len(rollouts)):
        if rollouts[i][0] < CRASH_ALT:
            print("Found failure!")
            weighted_sum += ws[i].item()
    return weighted_sum / len(rollouts)

In [16]:
print(qs[0])

(Array([ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, -1.6508633e-05,
        0.0000000e+00,  0.0000000e+00,  2.1600521e-03,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00],      dtype=float32), Array([1.e-10, 1.e-10, 1.e-10, 1.e-02, 1.e-10, 1.e-10, 3.e-02, 1.e-10,
       1.e-10, 1.e-10, 1.e-10, 1.e-10, 1.e-10, 1.e-10, 1.e-10, 1.e-10],      dtype=float32))


In [17]:
imp_sampl_fail_est(system, qs, num_rollouts=5)

Running Rollouts: 100%|██████████| 1/1 [03:05<00:00, 185.91s/it]

Finished proposal distribution likelihood
[0.0012463949, 0.26914635, 0.00026125857, 0.030197384, 0.0007101744]
Found failure!


0.0001420348766259849